# Train one agro-ai crop expert on Kaggle

Before running:

1. **Settings → Accelerator:** GPU (T4 or P100). **Internet:** on (to fetch the code).
2. **Add Input** the datasets the crop needs. They are used in place from `/kaggle/input`, nothing is downloaded:

| Crop | Inputs |
|---|---|
| cassava | competition `cassava-leaf-disease-classification` (accept its rules first) |
| maize | `smaranjitghose/corn-or-maize-leaf-disease-dataset`, `vipoooool/new-plant-diseases-dataset` |
| tomato, potato, apple | `vipoooool/new-plant-diseases-dataset` (optionally `nirmalsankalana/plant-diseases-training-dataset`, `salmasyed1360/plant-diseases-100k-labelled-images`) |
| tea | `shashwatwork/identifying-disease-in-tea-leafs` |
| coffee | `nirmalsankalana/rocole-a-robusta-coffee-leaf-images-dataset` |
| cashew | `olaidegabriel/cashew-disease-detection-dataset` |

3. Make the code available: either set `REPO_URL` below to the GitHub repo, or upload the repo folder as a private Kaggle dataset and add it as an input.

Outputs land in `/kaggle/working/models/` (download them from the notebook's Output tab).

In [ ]:
CROP = "cassava"
REPO_URL = ""                              # e.g. "https://github.com/<you>/agro-ai.git"
CODE_INPUT = "/kaggle/input/agro-ai-code"  # used when REPO_URL is empty
EXTRA_TRAIN_ARGS = "--mixed-precision"      # e.g. "--coord-attention --epochs-finetune 20"

In [ ]:
import os, shutil, subprocess, sys

CODE = "/kaggle/working/agro-ai"
if not os.path.isdir(CODE):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
    else:
        shutil.copytree(CODE_INPUT, CODE)
os.chdir(CODE)
sys.path.insert(0, CODE)
!pip install -q -e . --no-deps
!python dataset_factory.py --list-sources

In [ ]:
# Only sources attached as inputs are found; --no-download makes a missing one fail loudly.
!python dataset_factory.py --crops {CROP} --no-download --workers 4

In [ ]:
import json, pathlib

DATA = pathlib.Path("/kaggle/working/agro-data/processed") / CROP
report = json.loads((DATA / "report.json").read_text())
print("images:", report["total"], "| duplicates removed:", report["duplicates_removed"], "| rejected:", report["rejected"])
for label, c in sorted(report["classes"].items()):
    print(f"{label:<40} train={c.get('train', 0):>6} val={c.get('val', 0):>5} test={c.get('test', 0):>5} "
          f"field={c.get('domain_field', 0):>6} lab={c.get('domain_lab', 0):>6}")
print("\n".join(report["warnings"]) or "no warnings")

In [ ]:
!python -m agro_ai.train.train --data {DATA} {EXTRA_TRAIN_ARGS}

In [ ]:
manifest = json.loads(pathlib.Path("/kaggle/working/models/manifest.json").read_text())
latest = [m for m in manifest["models"] if m["name"] == CROP][-1]
print(json.dumps({k: latest[k] for k in ("name", "version", "recommended", "metrics")}, indent=2))
for f in latest["files"]:
    print(f"{f['variant']:<5} {f['bytes'] / 1e6:6.2f} MB  {f['path']}")

In [ ]:
# Keep the output small: drop the processed image copies, keep only models/.
shutil.rmtree("/kaggle/working/agro-data", ignore_errors=True)
shutil.rmtree(CODE, ignore_errors=True)